# 12.06 · Docker and Docker Compose

**Module 12 · Production & Deployment** · Part 2 (Pro)

| | |
|---|---|
| ⏱️ Estimated time | 3–4 h |
| 📋 Prerequisites | Lesson 12.05 (durable state). Docker Desktop for the 🐳 cells (optional: everything else runs without it). |
| 🔑 Keys | none (the container runs in DEMO mode; LIVE needs only `GROQ_API_KEY` in `.env`) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-16 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Explain** what a container image adds to "it works on my machine": one tested set of versions, a clean filesystem, one start command.
2. **Read** a production Dockerfile line by line: multi-stage build, the lock file, a non-root user, a health check, exec-form `CMD`.
3. **Keep secrets and junk out** of the image with an allowlist `.dockerignore`, and configure the container from the environment.
4. **Separate liveness from readiness** (`/ok` vs `/ready`) and see why an orchestrator needs both.
5. **Shut down gracefully**: on `SIGTERM`, refuse new runs, let active ones finish, save the rest so clients can retry.
6. **Run** the service with Postgres in Docker Compose: start order with health checks, a volume, restarts that keep every thread.

## 📚 Official reading (verified 2026-10-16)

| Page | Read it for |
|---|---|
| [Dockerfile reference](https://docs.docker.com/reference/dockerfile/) | `FROM … AS`, `COPY --from`, `USER`, `HEALTHCHECK`, exec-form `CMD` |
| [Multi-stage builds](https://docs.docker.com/build/building/multi-stage/) | Build in one stage, ship only the result |
| [Build context and `.dockerignore`](https://docs.docker.com/build/concepts/context/) | What gets sent to the builder; the `<Dockerfile>.dockerignore` naming |
| [Compose: startup order](https://docs.docker.com/compose/how-tos/startup-order/) | `depends_on` with `condition: service_healthy` and `restart: true` |
| [Compose file reference: services](https://docs.docker.com/reference/compose-file/services/) | `env_file` with `required: false`, `read_only`, `tmpfs`, `stop_grace_period`, `profiles` |
| [python on Docker Hub](https://hub.docker.com/_/python) · [postgres on Docker Hub](https://hub.docker.com/_/postgres) | The `3.12-slim` and `17-alpine` tags; Postgres environment variables and data directory |
| [Uvicorn settings](https://uvicorn.dev/settings/) | `--timeout-graceful-shutdown`, `--proxy-headers` |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langgraph>=1.2.14,<2" "langgraph-checkpoint-sqlite>=3.1.1,<4" "fastapi>=0.141.1,<0.142" "uvicorn>=0.54.0,<1" "pydantic-settings>=2.16.0,<3" "httpx>=0.28.1,<1" "pyjwt[crypto]>=2.15.1,<3" "langgraph-checkpoint-postgres>=3.2.0,<4" "psycopg[binary]>=3.3.6,<4" "prometheus-client>=0.26.0,<1"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("fastapi", "uvicorn", "langgraph-checkpoint-postgres", "prometheus-client",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/12-production-and-deployment/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY', 'LANGSMITH_API_KEY')
LANGSMITH_PROJECT = 'm12-docker'
# Setup 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
import os

from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_LANGSMITH_KEY = is_set("LANGSMITH_API_KEY")
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m12-<lesson>: one tracing project per lesson

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")
if not HAVE_LANGSMITH_KEY:
    print("⚠️ No LANGSMITH_API_KEY: 🌐 LangSmith cells will print ⏭️ Skipped; offline twins run instead.")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")          # Chroma (Project 09 LIVE FAQ index): no usage telemetry.
# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# fastembed (Project 09 LIVE FAQ index) imports tqdm.auto; without ipywidgets it warns once. Progress bars still work.
warnings.filterwarnings("ignore", message="IProgress not found")

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table (one comparison cell per Part)."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

In [ ]:
# 🐳 Is Docker here and running? (no tokens) The 🐳 cells below print ⏭️ Skipped without it.
import shutil
import subprocess

def docker(*args, timeout=600, check=True):
    """Run a docker command from the repository root and return its output (stdout + stderr)."""
    result = subprocess.run(["docker", *args], cwd=REPO_ROOT, capture_output=True, text=True, timeout=timeout)
    if check and result.returncode != 0:
        raise RuntimeError(f"docker {' '.join(args)} failed:\n{result.stdout}{result.stderr}")
    return (result.stdout + result.stderr).strip()

HAVE_DOCKER = False
if shutil.which("docker"):
    try:
        print("Docker server:", docker("version", "--format", "{{.Server.Version}}", timeout=20))
        print(docker("compose", "version", timeout=20))
        HAVE_DOCKER = True
    except (RuntimeError, subprocess.TimeoutExpired):
        print("⏭️ Docker is installed but not running: start Docker Desktop, then re-run this cell.")
else:
    print("⏭️ Skipped: no docker command (install Docker Desktop for the 🐳 cells).")

---
## Part 1 · What the image adds

Running `uvicorn apps.m12_service.main:app` works on *your* machine because your env has the right packages. A server, a teammate or CI starts from nothing. An **image** freezes four things:

| Without an image | With the image |
|---|---|
| "Install the course env first" | The venv is inside, built from `constraints/constraints-2026-10-16.txt` (the versions the tests passed with) |
| Your OS, your Python | `python:3.12-slim` (Debian), the same everywhere |
| Whatever files are in your clone (`.env`, notebooks, outputs) | Only what the allowlist lets in |
| "Run this command, with these variables" | `CMD` and `ENV` in the image; the rest from the environment |

The files are in [`apps/m12_service/`](../../apps/m12_service/): `Dockerfile`, `Dockerfile.dockerignore`, `compose.yaml`.

In [ ]:
# 🧪 Offline: read the Dockerfile's instructions (comments stripped).
APP_DIR = REPO_ROOT / "apps" / "m12_service"
for line in (APP_DIR / "Dockerfile").read_text(encoding="utf-8").splitlines():
    if line.strip() and not line.lstrip().startswith("#"):
        print(line)

**You should now see** two `FROM` lines (a `build` stage and the final one), `pip install -c constraints/… ".[llm,memory,service]"`, `USER app`, a `HEALTHCHECK` that calls `/ready`, and `CMD ["uvicorn", …]`.

### What just happened?

Each line is a decision:

- **Two stages.** The `build` stage creates `/opt/venv` and installs into it; the final stage copies only that folder. Build tools and pip caches never reach the image you ship.
- **Dependencies before code.** `pyproject.toml` and the lock are copied first, so Docker reuses the slow install layer until they change.
- **The lock file.** `-c constraints/…` installs exactly the tested versions; the `pyproject.toml` ranges still say what's allowed.
- **Not root.** `USER app` (uid 10001): a bug in the app can't rewrite the image's system files.
- **Exec-form `CMD`** (a JSON list): uvicorn is process 1 and receives `SIGTERM` itself. The shell form (`CMD uvicorn …`) puts `/bin/sh` in between, which doesn't forward signals: every `docker stop` waits 10 s and then kills.
- **`HEALTHCHECK`** with Python, since the slim image has no `curl`.

---
## Part 2 · What goes into the build (and what must not)

`docker build` sends the **build context** (here: the repository root) to the builder. Our repo has notebooks, datasets, outputs, maybe a `.env` with keys. `Dockerfile.dockerignore` (next to the Dockerfile, so it only applies to this image) is an **allowlist**: ignore everything (`*`), then let back in the few paths the image needs.

In [ ]:
# 🧪 Offline: how big is the context with and without the allowlist?
def size(path):
    if path.is_file():
        return path.stat().st_size
    return sum(p.stat().st_size for p in path.rglob("*") if p.is_file() and "__pycache__" not in p.parts)

allow = [line[1:].rstrip("/") for line in (APP_DIR / "Dockerfile.dockerignore").read_text(encoding="utf-8").splitlines()
         if line.startswith("!")]
skip_dirs = {".git", "workspace", ".venv", "node_modules"}
everything = sum(p.stat().st_size for p in REPO_ROOT.rglob("*")
                 if p.is_file() and not skip_dirs & set(p.relative_to(REPO_ROOT).parts))
rows = [{"path": a, "KB": round(size(REPO_ROOT / a) / 1024)} for a in allow]
table(rows, ["path", "KB"])
sent = sum(r["KB"] for r in rows)
print(f"\nallowlist sends ≈ {sent:,} KB of a ≈ {everything // 1024:,} KB clone (without .git)")
print(".env would be sent:", any(".env" == a or a.startswith(".env") for a in allow))

**You should now see** six paths, `src` being the largest, a context that is a small fraction of the clone, and `.env would be sent: False`.

### What just happened?

A denylist (`*.ipynb`, `.env`, …) leaks the first file nobody thought of; an allowlist fails safe. Keys reach the container at **run** time instead, from the environment (`env_file:` in Compose), never baked into a layer, where `docker history` or anyone with the image could read them.

> ⚠️ **Production note: build arguments are not secrets.** `--build-arg API_KEY=…` ends up in the image metadata. For a secret needed *during* the build (a private package index), use `RUN --mount=type=secret`.

---
## Part 3 · Configuration from the environment

The image is the same in every environment; only variables change (the "config" factor of the [Twelve-Factor App](https://12factor.net/config)). `ServiceSettings` reads `MASTERY_SERVICE_*` and **fails at startup** with a clear message if one is wrong, so a typo in Compose stops the container instead of breaking the first request.

In [ ]:
# 🧪 Offline: the same settings class, three environments.
from pydantic import ValidationError

from mastery.service.settings import ServiceSettings

environments = {
    "laptop": {},
    "compose": {"database_url": "postgresql://agent:agent@db:5432/agent", "log_format": "json", "release": "1.4.0"},
    "typo": {"max_concurrent": "four"},
}
for name, values in environments.items():
    try:
        s = ServiceSettings(_env_file=None, **values)
        print(f"{name:<8} ok    database={'set' if s.database() else 'memory'} log={s.log_format} release={s.release}")
    except ValidationError as exc:
        print(f"{name:<8} FAILS {exc.errors()[0]['loc'][0]}: {exc.errors()[0]['msg']}")

**You should now see** `laptop ok … database=memory`, `compose ok … database=set log=json release=1.4.0`, and `typo FAILS max_concurrent: Input should be a valid integer…`.

### What just happened?

`database_url` is a `SecretStr`: the line prints `set`, never the password. In `compose.yaml`, `environment:` sets the database URL and `env_file:` adds your `.env` *if it exists* (`required: false`), so the stack starts in DEMO mode on a fresh clone and switches to LIVE when you add `MASTERY_SERVICE_MODE=live` and a key.

---
## Part 4 · Alive is not the same as ready

An orchestrator (Compose, Kubernetes, a hosting platform) asks two questions:

| Probe | Endpoint | Question | When it fails |
|---|---|---|---|
| Liveness | `GET /ok` | Is the process stuck? | Restart the container |
| Readiness | `GET /ready` | Should it get traffic now? | Stop sending requests; don't restart |

`/ready` returns 503 while starting, while shutting down, or when the database doesn't answer. Restarting the API wouldn't fix a database outage, so that belongs in readiness, not liveness.

In [ ]:
# 🧪 Offline: readiness in three states.
import httpx

from mastery.service.app import create_app, serve_in_background
from mastery.service.client import bearer, stream_events
from mastery.service.desk import durable_support_manager

class Service:
    """Start the app in a background thread; stop it at the end of the `with` block (a 'container')."""
    def __init__(self, build=durable_support_manager, **settings):
        self.app = create_app(build, ServiceSettings(_env_file=None, **settings))
    def __enter__(self):
        self.url, self.stop = serve_in_background(self.app)
        self.client = httpx.Client(base_url=self.url, timeout=30)
        return self.client
    def __exit__(self, *exc):
        self.client.close()
        self.stop()

async def database_down():
    raise ConnectionError("connection refused")

with Service(release="1.4.0") as c:
    print("healthy  :", c.get("/ok").status_code, c.get("/ready").json())

**You should now see** `healthy  : 200 {'ready': True, 'problems': [], 'release': '1.4.0'}`.

In [ ]:
# 🧪 Offline: break the database, then start draining; liveness stays 200 throughout.
service = Service(release="1.4.0")
with service as c:
    manager = service.app.state.manager
    manager.store.ping = database_down
    print("db down  :", c.get("/ok").status_code, c.get("/ready").status_code, c.get("/ready").json()["problems"])
    del manager.store.ping                      # back to the real method
    manager.draining = True
    print("draining :", c.get("/ok").status_code, c.get("/ready").status_code, c.get("/ready").json()["problems"])
    tid = c.post("/threads", headers=bearer("demo-customer")).json()["thread_id"]
    refused = c.post(f"/threads/{tid}/runs", json={"message": "hello"}, headers=bearer("demo-customer"))
    print("new run  :", refused.status_code, refused.headers.get("retry-after"), refused.json()["detail"])
    manager.draining = False

**You should now see** `db down  : 200 503 ['database: ConnectionError']`, `draining : 200 503 ['shutting down']`, and `new run  : 503 5 the service is shutting down; retry in a few seconds`.

### What just happened?

Liveness answered 200 every time: the process was fine. Readiness told the load balancer to send traffic elsewhere, and a run submitted during the drain got **503 with `Retry-After`**, which a client (or a proxy) retries on another replica. Compare 429 (you, slow down) with 503 (we, temporarily unavailable).

---
## Part 5 · Graceful shutdown

Every deploy stops the old container. The chain, with the numbers in our files:

| Time | What happens | Where it's set |
|---|---|---|
| 0 s | `docker stop` (or a deploy) sends `SIGTERM` to process 1, uvicorn | exec-form `CMD` |
| 0 s | uvicorn stops accepting connections and waits for open ones (streams) | |
| ≤ 10 s | uvicorn cancels whatever is still open, then runs the app's shutdown | `--timeout-graceful-shutdown 10` |
| ≤ 20 s | `RunManager.shutdown`: refuse new runs, let active ones finish, then save the rest as `error` with "please send it again" | `MASTERY_SERVICE_DRAIN_S=10` |
| 30 s | Docker gives up and sends `SIGKILL` | `stop_grace_period: 30s` |

The rule: each limit fits inside the next one. A run killed by `SIGKILL` is the crash case from Lesson 12.05 (`recover()` closes it at the next start); a run cut off by the drain is saved properly right away.

In [ ]:
# 🧪 Offline: drain with simulated model latency (0.3 s per model call, labelled as such).
import asyncio
import time

from mastery.service.desk import support_manager

FAQ = "How do I rotate an API key?"
rows, messages = [], {}
for drain_s in (0.1, 10.0):
    manager = support_manager(ServiceSettings(_env_file=None), delay_s=0.3)
    runs = [await manager.submit(thread_id=f"t-{drain_s}-{i}", user_id="C-1002", input={"messages": [("user", FAQ)]})
            for i in range(4)]
    started = time.perf_counter()
    cut = await manager.shutdown(drain_s=drain_s)
    if cut:
        messages[drain_s] = (await manager.store.get_run(cut[0]))["error"]
    rows.append({"drain_s": drain_s, "finished": sum(r.status == "success" for r in runs), "cut off": len(cut),
                 "took_s": round(time.perf_counter() - started, 2),
                 "saved status": (await manager.store.get_run(runs[0].run_id))["status"]})
table(rows, ["drain_s", "finished", "cut off", "took_s", "saved status"])
print("\ncut-off message:", messages)

**You should now see** two rows: with `drain_s=0.1`, all 4 runs cut off in about 0.1 s and saved as `error`; with `drain_s=10`, all 4 finished (in about 1 s) and none were cut off. The cut-off runs carry `the service shut down during this run; please send it again`.

### What just happened?

A longer drain trades deploy speed for fewer interrupted users. Size it from your run durations (Lesson 12.01's duration SLO: 95% under 60 s would argue for a longer drain, or for clients that resume). The interrupted run is saved with a terminal status, so its stream ends and the client knows to resend.

---
## Part 6 · 🐳 Build the image

The first build downloads the base image and installs the packages (a few minutes, about 1 GB of downloads the first time). Later builds reuse the cached layers in seconds unless `pyproject.toml` or the lock changed. The same command in a terminal, from the repository root:

```bash
docker build -f apps/m12_service/Dockerfile -t mastery-m12-service:dev .
```

In [ ]:
# 🐳 Build (no tokens; several minutes the first time).
if HAVE_DOCKER:
    started = time.perf_counter()
    docker("build", "-f", "apps/m12_service/Dockerfile", "-t", "mastery-m12-service:dev", ".", timeout=1800)
    print(f"built in {time.perf_counter() - started:.0f} s")
    print(docker("image", "ls", "mastery-m12-service:dev", "--format", "{{.Repository}}:{{.Tag}}  {{.Size}}"))
    print("runs as      :", docker("run", "--rm", "--entrypoint", "id", "mastery-m12-service:dev"))
    listing = docker("run", "--rm", "--entrypoint", "ls", "mastery-m12-service:dev", "-a", "/app")
    print("/app         :", listing.split())
else:
    print("⏭️ Skipped: Docker is not available (see the check at the top).")

**You should now see** the build time, an image of a few hundred MB, `uid=10001(app)`, and `/app` holding only `COURSE_OUTLINE.md`, `apps`, `data` and `pyproject.toml`: no `.env`, no notebooks.

---
## Part 7 · 🐳 The whole stack with Compose

`compose.yaml` runs two services: `api` (our image) and `db` (`postgres:17-alpine` with a named volume). `depends_on: condition: service_healthy` makes Compose wait until `pg_isready` passes, not merely until the database container exists. In a terminal:

```bash
docker compose -f apps/m12_service/compose.yaml up --build -d
docker compose -f apps/m12_service/compose.yaml ps
```

The cells below do the same, then prove the state survives restarting the API container.

In [ ]:
# 🐳 Start the stack and wait for /ready (no tokens: DEMO mode unless your .env says live).
COMPOSE = ["compose", "-f", "apps/m12_service/compose.yaml"]
API = "http://127.0.0.1:8000"

def wait_ready(seconds=90):
    deadline = time.monotonic() + seconds
    while time.monotonic() < deadline:
        try:
            r = httpx.get(f"{API}/ready", timeout=2)
            if r.status_code == 200:
                return r.json()
        except httpx.HTTPError:
            pass
        time.sleep(1)
    raise TimeoutError("the API did not become ready; see: docker compose -f apps/m12_service/compose.yaml logs api")

if HAVE_DOCKER:
    print((docker(*COMPOSE, "up", "--build", "-d", timeout=1800).splitlines() or ["started"])[-1])
    print("ready:", wait_ready())
    print(docker(*COMPOSE, "ps", "--format", "{{.Service}}  {{.Status}}"))
else:
    print("⏭️ Skipped: Docker is not available.")

**You should now see** `ready: {'ready': True, 'problems': [], 'release': 'dev'}` and both services `Up …`: `db` `(healthy)`, `api` `(healthy)` or still `(health: starting)` for its first 20 s.

> If port 8000 is busy (another uvicorn from Lesson 12.04?), stop that first, or change the left side of `"127.0.0.1:8000:8000"`.

In [ ]:
# 🐳 A paused refund, a restart of the API container, then the manager approves.
REFUND = "I'm customer C-1002. I want a refund of $199 for INV-2001, we moved to another vendor."
if HAVE_DOCKER:
    with httpx.Client(base_url=API, timeout=30) as c:
        tid = c.post("/threads", headers=bearer("demo-customer")).json()["thread_id"]
        run_id = c.post(f"/threads/{tid}/runs", json={"message": REFUND}, headers=bearer("demo-customer")).json()["run_id"]
        print("before restart:", stream_events(c, run_id, "demo-customer")[-1].event)
    print((docker(*COMPOSE, "restart", "api").splitlines() or ["restarted"])[-1])
    wait_ready()
    with httpx.Client(base_url=API, timeout=30) as c:
        print("after restart :", c.get(f"/runs/{run_id}", headers=bearer("demo-customer")).json()["status"])
        resumed = c.post(f"/threads/{tid}/runs", json={"resume": "approve"}, headers=bearer("demo-manager")).json()
        print("approved      :", stream_events(c, resumed["run_id"], "demo-manager")[-1].data["answer"][:60])
    logs = docker(*COMPOSE, "logs", "--no-log-prefix", "api")
    print("\nlast JSON log line:", [line for line in logs.splitlines() if line.startswith("{")][-1][:200])
else:
    print("⏭️ Skipped: Docker is not available.")

**You should now see** `before restart: interrupt`, the restart, `after restart : interrupted`, `approved      : Refunded $199.00 for INV-2001 …`, and a JSON log line (Lesson 12.07 reads these).

### What just happened?

The API container was replaced and nothing was lost: the thread, its owner, the paused approval and the run history live in the `db` container's volume. That is the property that makes deploys boring.

In [ ]:
# 🐳 Stop the stack. `down` keeps the volume (the database); `down -v` deletes it too.
if HAVE_DOCKER:
    started = time.perf_counter()
    docker(*COMPOSE, "down", timeout=120)
    print(f"stopped in {time.perf_counter() - started:.1f} s (graceful: no 10 s SIGKILL wait)")
    print("volume kept:", "m12-service_pgdata" in docker("volume", "ls", "--format", "{{.Name}}"))
else:
    print("⏭️ Skipped: Docker is not available.")

**You should now see** a stop time of a few seconds and `volume kept: True`. Run `docker compose -f apps/m12_service/compose.yaml down -v` when you want a fresh database.

---
## ⚠️ Pitfalls and troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `docker stop` always takes 10 s | Shell-form `CMD`: `/bin/sh` doesn't forward `SIGTERM` | Exec form `CMD ["uvicorn", …]` |
| API exits at start: `connection refused` to `db` | It started before Postgres was ready | `depends_on: condition: service_healthy` + a `healthcheck` on `db` |
| `localhost:5432` refused *inside* the API container | `localhost` is the container itself | Use the service name: `db:5432` |
| Every build reinstalls all packages | Code copied before the install step | Copy `pyproject.toml` and the lock first |
| `.env` or notebooks inside the image | Denylist (or no `.dockerignore`) | Allowlist; check with `docker run --rm --entrypoint ls IMAGE -a /app` (arguments after the image go to the entrypoint) |
| `Read-only file system` error | `read_only: true` and the app writes a file (e.g. SQLite) | Postgres for state; a `tmpfs` or a volume for the path that must be writable |
| `port is already allocated` | Something else listens on 8000 | Stop it, or change the host port |
| Database empty after `down` | You ran `down -v` | `down` without `-v`; back up before deleting volumes |

## ⚠️ Production notes

> ⚠️ **Production note: pin the base image.** `python:3.12-slim` moves with every patch release, which is good for security fixes and bad for reproducibility. Pin a digest (`python:3.12-slim@sha256:…`) and let a bot (Dependabot, Renovate) propose updates, so a base change goes through CI like any other change.

> ⚠️ **Production note: scan what you ship.** `docker scout cves mastery-m12-service:dev` (or Trivy) lists known vulnerabilities in the OS packages and Python wheels. Rebuild regularly even if your code didn't change.

> ⚠️ **Production note: one process per container, many containers per service.** Scale by running more `api` replicas behind a proxy, all on the same Postgres, rather than many uvicorn workers in one container: the platform can then restart, place and count them.

## 🏋️ Exercises

1. **(Easy)** Tag a release: `docker tag mastery-m12-service:dev mastery-m12-service:1.4.1`, start the stack with `RELEASE=1.4.1` and `--no-build`, and confirm the label in `/ready`.
2. **(Medium)** Make the API fail its health check on purpose (stop `db` with `docker compose … stop db`). What do `docker compose ps`, `/ok` and `/ready` say? Start `db` again: what happens to `api`?
3. **(Medium)** Run two API replicas on one database: remove the fixed host port and use `docker compose … up -d --scale api=2`. Which parts of the service are safe with two replicas, and which are not (hint: the semaphore and the budget ledger live in memory)?
4. **(Stretch)** Pin `python:3.12-slim` by digest (`docker buildx imagetools inspect python:3.12-slim`) and measure the image size with `python:3.12-alpine` instead. What breaks, and why are wheels the reason?

<details>
<summary><b>Show solutions</b></summary>

**1.** `RELEASE=1.4.1 docker compose -f apps/m12_service/compose.yaml up -d --no-build` (PowerShell: `$env:RELEASE='1.4.1'` first). `RELEASE` picks the image tag (`mastery-m12-service:${RELEASE:-dev}`) and sets `MASTERY_SERVICE_RELEASE`, so `/ready` shows `"release": "1.4.1"`. The same bytes now have a release name; rolling back is starting the previous tag (Lesson 12.09).

**2.** `ps` shows `api` turning `unhealthy` after three failed checks; `/ok` stays 200 and `/ready` returns 503 `database: OperationalError`. When `db` is healthy again, `restart: true` in `depends_on` restarts `api` and it reconnects.

**3.** Safe: everything in Postgres (threads, owners, runs, events, idempotency keys, spend rows, checkpoints). Not safe: the concurrency limit and the in-memory `SpendLedger` are per replica (two replicas allow twice the concurrency and each checks only its own spend), and a run's live stream is followed from the replica running it (another replica replays it from the store with a delay of `flush_every_s`). Fixes: per-replica limits sized for the total, budgets checked against the database (`spend_totals`), sticky sessions or a shared broker for live streams.

**4.** Alpine uses musl instead of glibc, so many binary wheels (`psycopg[binary]`, `orjson`, `cryptography`) don't install as-is and pip tries to compile from source (or fails). Slim keeps manylinux wheels working; the size difference is small next to the Python packages.
</details>

## 🔎 Docs drill (5 min)

In Compose, what is the difference between `condition: service_started` and `condition: service_healthy` in `depends_on`, and what does adding `restart: true` change?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. Open [Control startup and shutdown order](https://docs.docker.com/compose/how-tos/startup-order/).
2. `service_started` waits only until the dependency's container is **running**; `service_healthy` waits until its `healthcheck` **passes**.
3. `restart: true` restarts the dependent service when Compose restarts or updates the dependency (for example `docker compose restart db`).

</details>

## ✅ Checkpoint

- [ ] I can explain each line of the Dockerfile, including why there are two stages and why `CMD` is a JSON list.
- [ ] The allowlist keeps `.env`, notebooks and outputs out of the build context.
- [ ] I know which settings come from the environment and that a bad value stops the container at startup.
- [ ] I can say when `/ok` and `/ready` disagree, and what an orchestrator does with each.
- [ ] Shutdown drains runs, and the time limits fit inside each other (10 s, 10 s, 30 s).
- [ ] (🐳) The stack started healthy, kept a paused approval across an API restart, and stopped in seconds.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Check the [python](https://hub.docker.com/_/python) and [postgres](https://hub.docker.com/_/postgres) tags on Docker Hub and log in [`docs/research-log.md`](../../docs/research-log.md): the current `3.12-slim` Debian release (2026-10-16: trixie), the latest Postgres major (2026-10-16: 18, whose data directory moved to `/var/lib/postgresql`; this course stays on 17), and your image's size.

## Summary

- An image freezes the versions (lock file), the OS and the start command; configuration comes from the environment.
- Multi-stage, dependencies first, non-root, allowlisted context, exec-form `CMD`, a health check.
- Liveness (`/ok`) restarts stuck processes; readiness (`/ready`) routes traffic away from ones that can't serve.
- Graceful shutdown: stop accepting, drain, save what's left; every timeout fits inside the next.
- Compose: start order by health, a volume for state, restarts that lose nothing.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 12-07-observability-and-operations.ipynb** → [`12-07-observability-and-operations.ipynb`](12-07-observability-and-operations.ipynb).

[Module 12 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)